In [3]:
#Import packages
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [10]:

#Load the PBD file
fixer = PDBFixer(filename='/home/users/ys472/ying_Project/kinase_inhibitor_design/structures/processed/pkn2_dephos-complex.pdb')

/home/users/ys472/miniconda3/envs/AmberTools22/lib/python3.10/site-packages/openmm/app/internal/pdbstructure.py:537: UserWarning: WARNING: duplicate atom (ATOM      8  HA  SER A 644     -20.770 -31.573  -3.288  1.00  0.00           H  , ATOM      7  HA  SER A 644     -20.770 -31.573  -3.288  1.00  0.00           H  )
  warnings.warn("WARNING: duplicate atom (%s, %s)" % (atom, old_atom._pdb_string(old_atom.serial_number, atom.alternate_location_indicator)))
/home/users/ys472/miniconda3/envs/AmberTools22/lib/python3.10/site-packages/openmm/app/internal/pdbstructure.py:537: UserWarning: WARNING: duplicate atom (ATOM     10  HB2 SER A 644     -18.541 -32.515  -3.896  1.00  0.00           H  , ATOM      9  HB2 SER A 644     -18.541 -32.515  -3.896  1.00  0.00           H  )
  warnings.warn("WARNING: duplicate atom (%s, %s)" % (atom, old_atom._pdb_string(old_atom.serial_number, atom.alternate_location_indicator)))
/home/users/ys472/miniconda3/envs/AmberTools22/lib/python3.10/site-packages/op

In [5]:
# Add missing residues and atoms
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.0)

In [6]:
# Save cleaned PDB
with open('pkn2.cleaned.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

In [11]:
# Removes waters, non-standard heteroatoms, and any unwanted molecules.
from Bio.PDB import PDBParser, PDBIO, Select

class KeepChainAandB(Select):
    def accept_chain(self, chain):
        return chain.id in ['A', 'B']  # Keep only chains A and B

parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', 'pkn2.cleaned.pdb')

io = PDBIO()
io.set_structure(structure)
io.save('pkn2.cleaned.filtered.pdb', select=KeepChainAandB())



In [9]:
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = f"{chain.id}{residue.id[1]}"
            print(res_id, residue.resname)

A1 SER
A2 MET
A3 SER
A4 GLN
A5 GLN
A6 ARG
A7 PHE
A8 GLN
A9 PHE
A10 ASN
A11 LEU
A12 GLN
A13 ASP
A14 PHE
A15 ARG
A16 CYS
A17 CYS
A18 ALA
A19 VAL
A20 LEU
A21 LEU
A22 ARG
A23 GLY
A24 HIS
A25 PHE
A26 GLY
A27 LYS
A28 VAL
A29 LEU
A30 LEU
A31 ALA
A32 GLU
A33 TYR
A34 LYS
A35 ASN
A36 THR
A37 ASN
A38 GLU
A39 MET
A40 PHE
A41 ALA
A42 ILE
A43 LYS
A44 ALA
A45 LEU
A46 LYS
A47 LYS
A48 GLY
A49 ASP
A50 ILE
A51 VAL
A52 ALA
A53 ARG
A54 ASP
A55 GLU
A56 VAL
A57 ASP
A58 SER
A59 LEU
A60 MET
A61 CYS
A62 GLU
A63 LYS
A64 ARG
A65 ILE
A66 PHE
A67 GLU
A68 THR
A69 VAL
A70 ASN
A71 SER
A72 VAL
A73 ARG
A74 HIS
A75 PRO
A76 PHE
A77 LEU
A78 VAL
A79 ASN
A80 LEU
A81 PHE
A82 ALA
A83 CYS
A84 PHE
A85 GLN
A86 THR
A87 LYS
A88 GLU
A89 HIS
A90 VAL
A91 CYS
A92 PHE
A93 VAL
A94 MET
A95 GLU
A96 TYR
A97 ALA
A98 ALA
A99 GLY
A100 GLY
A101 ASP
A102 LEU
A103 MET
A104 MET
A105 HIS
A106 ILE
A107 HIS
A108 THR
A109 ASP
A110 VAL
A111 PHE
A112 SER
A113 GLU
A114 PRO
A115 ARG
A116 ALA
A117 VAL
A118 PHE
A119 TYR
A120 ALA
A121 ALA
A122 CYS
A123 VAL
A

In [12]:
#Script for checking if the pdb satisfies osprey K* requirement
from Bio.PDB import PDBParser
from collections import defaultdict

pdb_path = "pkn2.cleaned.filtered.pdb"
parser = PDBParser(QUIET=True)
structure = parser.get_structure("model", pdb_path)

# Track issues
missing_backbone = []
duplicate_atoms = defaultdict(list)
seen_atoms = set()

for model in structure:
    for chain in model:
        for residue in chain:
            res_id = f"{chain.id}{residue.id[1]}"
            atom_names = set()
            for atom in residue:
                key = (chain.id, residue.id[1], atom.name)
                if key in seen_atoms:
                    duplicate_atoms[res_id].append(atom.name)
                else:
                    seen_atoms.add(key)
                    atom_names.add(atom.name)
            # Check for missing backbone atoms
            required_backbone = {'N', 'CA', 'C', 'O'}
            if not required_backbone.issubset(atom_names):
                missing_backbone.append(res_id)

# Report
print("✅ Basic structure read successful.")
if duplicate_atoms:
    print("⚠️ Duplicate atoms found:")
    for res, atoms in duplicate_atoms.items():
        print(f"  {res}: {atoms}")
else:
    print("✅ No duplicate atoms.")

if missing_backbone:
    print("⚠️ Residues missing backbone atoms:")
    print(missing_backbone)
else:
    print("✅ All residues have full backbone.")

# Check for alternate conformers or insertion codes
altloc_issues = [
    (residue.id, residue.get_resname())
    for model in structure
    for chain in model
    for residue in chain
    if residue.id[2] != ' '  # insertion code or altloc
]

if altloc_issues:
    print("⚠️ Residues with alternate location or insertion code:")
    for rid, name in altloc_issues:
        print(f"  {rid}: {name}")
else:
    print("✅ No altLoc or insertion code issues.")


✅ Basic structure read successful.
✅ No duplicate atoms.
✅ All residues have full backbone.
✅ No altLoc or insertion code issues.
